# NLP Fundamentals

Natural Language Processing powers search engines, chatbots, translation, sentiment analysis, and LLMs. This notebook covers the progression from bag-of-words to transformer embeddings.

**Topics:** Text preprocessing, TF-IDF, word embeddings, sentence transformers, text classification pipeline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import string
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

# Sample corpus
texts = [
    "This product is absolutely amazing! Best purchase ever.",
    "Terrible quality, broke after one day. Complete waste of money.",
    "Good value for money, arrived on time. Happy with the purchase.",
    "Worst customer service I've ever experienced. Will not buy again.",
    "Decent product but shipping was slow. Would buy again though.",
    "Fantastic! Exceeded my expectations. Highly recommend!",
    "Disappointed. The product looks nothing like the photos.",
    "Great product, fast delivery, excellent packaging.",
]
labels = [1, 0, 1, 0, 1, 1, 0, 1]  # 1=positive, 0=negative

print('Sample texts and labels:')
for text, label in zip(texts[:4], labels[:4]):
    print(f'  [{"POS" if label else "NEG"}] {text[:60]}...')

## 1. Text Preprocessing

In [ ]:
def preprocess_text(text):
    """Standard text preprocessing pipeline."""
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

# Simple tokenizer (use spaCy/NLTK in production)
stop_words = {'the','a','an','is','it','this','and','or','of','to','was','in','for','with','on'}

def tokenize(text, remove_stopwords=True):
    tokens = preprocess_text(text).split()
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

# Test preprocessing
sample = texts[0]
print('Original:', sample)
print('Preprocessed:', preprocess_text(sample))
print('Tokenized:', tokenize(sample))

# Vocabulary analysis
all_tokens = [t for text in texts for t in tokenize(text)]
freq = Counter(all_tokens).most_common(10)
print('\nTop 10 tokens:')
for token, count in freq:
    print(f'  {token}: {count}')

## 2. Bag-of-Words vs TF-IDF

In [ ]:
# Bag of Words: count occurrences
bow = CountVectorizer(max_features=100, stop_words='english')
X_bow = bow.fit_transform(texts)
print('BoW matrix shape:', X_bow.shape)
print('Vocabulary sample:', list(bow.vocabulary_.keys())[:10])

# TF-IDF: weight by rarity across documents
# TF(t,d) * IDF(t) = count(t,d)/total_terms(d) * log(N/df(t))
tfidf = TfidfVectorizer(max_features=100, stop_words='english', ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(texts)
print('\nTF-IDF matrix shape:', X_tfidf.shape)

# Show TF-IDF weights for first document
doc_idx = 0
feature_names = tfidf.get_feature_names_out()
tfidf_scores = X_tfidf[doc_idx].toarray().flatten()
top_features = sorted(zip(feature_names, tfidf_scores), key=lambda x: x[1], reverse=True)[:10]

print(f'\nTop TF-IDF features in doc 0: "{texts[0][:50]}..."')
for feat, score in top_features:
    print(f'  {feat:<25} {score:.4f}')

print('\nKey TF-IDF insight: common words across ALL docs get lower IDF weight')
print('→ unique words to a doc get higher weight')

## 3. Sentiment Classification Pipeline

In [ ]:
# Use a larger synthetic dataset
# In practice: IMDB dataset, Amazon reviews, Twitter sentiment

positive_phrases = [
    'great product amazing quality excellent',
    'fast delivery good value highly recommend',
    'fantastic exceeded expectations love',
    'perfect quality beautiful well made',
    'outstanding customer service very happy'
]
negative_phrases = [
    'terrible quality broke immediately waste money',
    'slow delivery missing parts disappointed',
    'awful poor quality not recommend',
    'defective arrived damaged return',
    'horrible customer service never again'
]

# Generate more samples
np.random.seed(42)
corpus = []
labels_large = []
for _ in range(200):
    if np.random.random() > 0.5:
        words = np.random.choice(positive_phrases).split()
        np.random.shuffle(words)
        corpus.append(' '.join(words) + ' ' + np.random.choice(['product', 'service', 'item', 'purchase']))
        labels_large.append(1)
    else:
        words = np.random.choice(negative_phrases).split()
        np.random.shuffle(words)
        corpus.append(' '.join(words) + ' ' + np.random.choice(['product', 'service', 'item', 'purchase']))
        labels_large.append(0)

X_tr, X_te, y_tr, y_te = train_test_split(corpus, labels_large, test_size=0.2, random_state=42)

# Pipeline: TF-IDF + Logistic Regression
sentiment_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=500, stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, C=1.0))
])

sentiment_pipeline.fit(X_tr, y_tr)
y_pred = sentiment_pipeline.predict(X_te)
print('Sentiment Classification Report:')
print(classification_report(y_te, y_pred, target_names=['Negative', 'Positive']))

# Test on new sentences
new_texts = [
    "This is amazing and I love it!",
    "Terrible experience, very disappointed.",
    "Average product, nothing special."
]
predictions = sentiment_pipeline.predict(new_texts)
probabilities = sentiment_pipeline.predict_proba(new_texts)

print('\nPredictions on new texts:')
for text, pred, prob in zip(new_texts, predictions, probabilities):
    sentiment = 'Positive' if pred else 'Negative'
    print(f'  [{sentiment} ({prob[pred]:.1%})] {text}')

## 4. Word Embeddings & Sentence Transformers

In [ ]:
# Word2Vec-style embeddings: similar words have similar vectors
# Demo without gensim (manual illustration)

print('Word2Vec concept demo:')
print('  king - man + woman ≈ queen')
print('  Paris - France + Italy ≈ Rome')
print('  Embeddings capture semantic relationships!')
print()

# Sentence Transformers (state of the art for semantic similarity)
print('Sentence Transformers (semantic embeddings):')
print('  Install: pip install sentence-transformers')
print()

sentence_transformer_code = '''
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')  # 80MB, fast

sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",   # semantically similar
    "Machine learning is powerful",  # different topic
]

embeddings = model.encode(sentences)  # (3, 384) — 384-dim vectors
similarity_matrix = cosine_similarity(embeddings)

print('Cosine similarity matrix:')
print(similarity_matrix.round(3))
# Sentences 0 and 1 are similar → high cosine similarity!
'''
print(sentence_transformer_code)

# Simulate what the output looks like
sim_matrix = np.array([
    [1.000, 0.834, 0.121],
    [0.834, 1.000, 0.098],
    [0.121, 0.098, 1.000],
])

import seaborn as sns
plt.figure(figsize=(6, 4))
sns.heatmap(sim_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
            xticklabels=['Cat on mat', 'Feline on rug', 'ML is powerful'],
            yticklabels=['Cat on mat', 'Feline on rug', 'ML is powerful'])
plt.title('Semantic Similarity Matrix\n(Sentence Transformers output)')
plt.tight_layout(); plt.show()